In [1]:
import pandas as pd 
import numpy as np 

df_full = pd.read_csv("D1.csv")
df_clean = pd.DataFrame()
df_full.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,D1,22/08/2025,19:30,Bayern Munich,RB Leipzig,6,0,H,3,0,...,1.98,1.88,1.98,1.93,1.99,1.93,1.90,1.86,2.07,1.92
1,D1,23/08/2025,14:30,Ein Frankfurt,Werder Bremen,4,1,H,2,0,...,1.83,2.03,2.02,1.91,1.91,2.03,1.83,1.93,1.91,2.06
2,D1,23/08/2025,14:30,Freiburg,Augsburg,1,3,A,0,3,...,1.93,1.93,1.97,1.95,1.97,1.93,1.90,1.86,2.03,1.96
3,D1,23/08/2025,14:30,Heidenheim,Wolfsburg,1,3,A,1,1,...,2.03,1.83,2.06,1.87,2.03,1.85,1.97,1.81,2.12,1.88
4,D1,23/08/2025,14:30,Leverkusen,Hoffenheim,1,2,A,1,1,...,1.98,1.88,1.97,1.95,1.98,2.02,1.88,1.86,2.00,1.97


In [2]:
df_full['Date'] = pd.to_datetime(df_full['Date'],format="%d/%m/%Y")
last_date_of_date = df_full['Date'].max()
df_full['daystoLast'] = (last_date_of_date-df_full['Date']).dt.days

In [3]:
team_map = {team:i for i,team in enumerate(sorted(df_full['HomeTeam'].unique()))}
outcome_map = {outcome:i for i,outcome in enumerate(sorted(df_full['FTR'].unique()))}
time_slot_map = {time_slot:i for i,time_slot in enumerate(sorted(df_full['Time'].unique()))}

In [4]:
def add_team_form_features(df):

    df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
    

    conditions_home = [df['FTR'] == 'H', df['FTR'] == 'D', df['FTR'] == 'A']
    conditions_away = [df['FTR'] == 'A', df['FTR'] == 'D', df['FTR'] == 'H']
    points = [3, 1, 0]

    df['HomePoints'] = np.select(conditions_home, points, default=0)
    df['AwayPoints'] = np.select(conditions_away, points, default=0)


    home_df = df[['Date', 'HomeTeam', 'HomePoints']].rename(
        columns={'HomeTeam': 'Team', 'HomePoints': 'Points'})
    away_df = df[['Date', 'AwayTeam', 'AwayPoints']].rename(
        columns={'AwayTeam': 'Team', 'AwayPoints': 'Points'})

    home_df['Original_Index'] = df.index
    home_df['Is_Home'] = True
    away_df['Original_Index'] = df.index
    away_df['Is_Home'] = False

    all_matches = pd.concat([home_df, away_df]).sort_values(by=['Date', 'Original_Index'])


    all_matches['Form_Points_Last_3'] = all_matches.groupby('Team')['Points'].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
    )

    all_matches['Form_Points_Last_3'] = all_matches['Form_Points_Last_3'].fillna(0)

    home_form = all_matches[all_matches['Is_Home'] == True].set_index('Original_Index')
    away_form = all_matches[all_matches['Is_Home'] == False].set_index('Original_Index')

    df['Home_Form_Pts_Last_3'] = df.index.map(home_form['Form_Points_Last_3'])
    df['Away_Form_Pts_Last_3'] = df.index.map(away_form['Form_Points_Last_3'])

    return df

In [5]:
def calculate_rolling_goal_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['FTHG'])
        else:
            goals.append(past_row['FTAG'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def calculate_rolling_shot_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['HST'])
        else:
            goals.append(past_row['AST'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def calculate_rolling_goal_avg(row, team_col, dataframe):
    team = row[team_col]
    current_date = row['Date']
    
    past_games = dataframe[
        ((dataframe['HomeTeam'] == team) | (dataframe['AwayTeam'] == team)) & 
        (dataframe['Date'] < current_date)
    ].sort_values('Date', ascending=False)
    
    goals = []
    for _, past_row in past_games.iterrows():
        if past_row['HomeTeam'] == team:
            goals.append(past_row['FTHG'])
        else:
            goals.append(past_row['FTAG'])
    
    last_5 = goals[:5]
    while len(last_5) < 5:
        last_5.append(0)
        
    return sum(last_5) / 5

def get_points(row, team_type):
    if row['FTR'] == 'D':
        return 1
    if team_type == 'Home':
        return 3 if row['FTR'] == 'H' else 0
    else:
        return 3 if row['FTR'] == 'A' else 0
    

In [6]:
def get_rolling_team_form(df, window=3):
    home_df = df[['Date', 'HomeTeam', 'HomePoints']].copy()
    home_df.columns = ['Date', 'Team', 'Points']
    home_df['Is_Home'] = True
    home_df['Original_Index'] = df.index

    away_df = df[['Date', 'AwayTeam', 'AwayPoints']].copy()
    away_df.columns = ['Date', 'Team', 'Points']
    away_df['Is_Home'] = False
    away_df['Original_Index'] = df.index

    all_matches = pd.concat([home_df, away_df]).sort_values(by=['Date', 'Original_Index'])

    all_matches['Form'] = all_matches.groupby('Team')['Points'].transform(
        lambda x: x.shift(1).rolling(window=window, min_periods=1).sum()
    )
    
    all_matches['Form'] = all_matches['Form'].fillna(0)

    home_form = all_matches[all_matches['Is_Home']].set_index('Original_Index')['Form']
    away_form = all_matches[~all_matches['Is_Home']].set_index('Original_Index')['Form']

    return home_form.reindex(df.index), away_form.reindex(df.index)

In [7]:
df_full['Home_Avg_Last5_Goal'] = df_full.apply(lambda row: calculate_rolling_goal_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Avg_Last5_Goal'] = df_full.apply(lambda row: calculate_rolling_goal_avg(row,'AwayTeam',df_full),axis=1)
df_full['Home_Avg_Last5_Shot'] = df_full.apply(lambda row: calculate_rolling_shot_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Avg_Last5_Shot'] = df_full.apply(lambda row: calculate_rolling_shot_avg(row,'AwayTeam',df_full),axis=1)
df_full['HomePoints'] = df_full.apply(lambda x: get_points(x, 'Home'), axis=1)
df_full['AwayPoints'] = df_full.apply(lambda x: get_points(x, 'Away'), axis=1)
df_full['Home_Last_3_Points'] = df_full.apply(lambda row:calculate_rolling_goal_avg(row,'HomeTeam',df_full),axis=1)
df_full['Away_Last_3_Points'] = df_full.apply(lambda row:calculate_rolling_goal_avg(row,'AwayTeam',df_full),axis=1)
df_full['Home_Form_Last_3'], df_full['Away_Form_Last_3'] = get_rolling_team_form(df_full, window=3)

In [8]:
#Date for Split
df_clean['date'] = df_full['Date']
#Data for Prediction
df_clean['timeSlot'] = df_full['Time'].map(time_slot_map)
df_clean['HomeTeam']=df_full['HomeTeam'].map(team_map)
df_clean['AwayTeam'] = df_full['AwayTeam'].map(team_map)
df_clean['homeGoal5GameAvgDiff'] = df_full['Home_Avg_Last5_Goal']-df_full['Away_Avg_Last5_Goal']
df_clean['homeShot5GameAvgDiff'] = df_full['Home_Avg_Last5_Shot'] - df_full['Away_Avg_Last5_Shot']
df_clean['homeTeamFormDiff'] = df_full['Home_Form_Last_3'] - df_full['Away_Form_Last_3']
df_clean['bookmakerHomeWinOdds'] = df_full['AvgH']
df_clean['bookmakerHomeDrawOdds'] = df_full['AvgD']
df_clean['bookmakerAwayWinOdds'] = df_full['AvgA']


#Predict Value
df_clean['homeWin'] = df_full['FTR'].map(outcome_map)

df_clean.head()

,date,timeSlot,HomeTeam,AwayTeam,homeGoal5GameAvgDiff,homeShot5GameAvgDiff,homeTeamFormDiff,bookmakerHomeWinOdds,bookmakerHomeDrawOdds,bookmakerAwayWinOdds,homeWin
0,2025-08-22,4,1,12,0.0,0.0,0.0,1.21,7.13,10.30,2
1,2025-08-23,0,3,16,0.0,0.0,0.0,1.69,4.00,4.46,2
2,2025-08-23,0,5,0,0.0,0.0,0.0,1.82,3.58,4.25,0
3,2025-08-23,0,7,17,0.0,0.0,0.0,3.05,3.51,2.20,0
4,2025-08-23,0,9,8,0.0,0.0,0.0,1.50,4.63,5.56,0


In [9]:
#Wheigt so that Data old Data is not as Relevant
df_clean['weight'] = np.exp(-df_full['daystoLast']/100)

In [10]:
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean = df_clean.sort_values('date').reset_index(drop=True)

In [12]:
features = ['timeSlot', 'HomeTeam', 'AwayTeam', 'homeGoal5GameAvgDiff','homeTeamFormDiff',
            'bookmakerHomeWinOdds', 'bookmakerHomeDrawOdds', 'bookmakerAwayWinOdds']
X = df_clean[features]
y = df_clean['homeWin']
weights = df_clean['weight']

In [20]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
modell = xgb.XGBClassifier(
    n_estimators=500,   
    max_depth=2,         
    learning_rate=0.05,   
    random_state=42,
    early_stopping_rounds=20, 
    objective='multi:softprob', 
    eval_metric='mlogloss'
)

In [21]:
tscv = TimeSeriesSplit(n_splits=20)
scores_acc = []
scores_logloss = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    w_train, w_test = weights.iloc[train_index], weights.iloc[test_index]
    
    modell.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_train, y_train), (X_test, y_test)], 
        sample_weight_eval_set=[w_train, w_test],
        verbose=False 
    )
    
    acc = modell.score(X_test, y_test)
    scores_acc.append(acc)
    

    results = modell.evals_result()

    final_fold_logloss = results['validation_1']['mlogloss'][-1] 
    scores_logloss.append(final_fold_logloss)
    
    print(f"Fold Accuracy: {acc:.4f} | Fold Log Loss: {final_fold_logloss:.4f}")


Fold Accuracy: 0.7778 | Fold Log Loss: 0.8838
Fold Accuracy: 0.6667 | Fold Log Loss: 0.8633
Fold Accuracy: 0.5556 | Fold Log Loss: 1.0609
Fold Accuracy: 0.6667 | Fold Log Loss: 0.9526
Fold Accuracy: 0.6667 | Fold Log Loss: 0.7807
Fold Accuracy: 0.4444 | Fold Log Loss: 1.1497
Fold Accuracy: 0.6667 | Fold Log Loss: 0.9704
Fold Accuracy: 0.7778 | Fold Log Loss: 0.5727
Fold Accuracy: 0.4444 | Fold Log Loss: 1.1620
Fold Accuracy: 0.6667 | Fold Log Loss: 0.8786
Fold Accuracy: 0.6667 | Fold Log Loss: 1.0535
Fold Accuracy: 0.3333 | Fold Log Loss: 1.2063
Fold Accuracy: 0.6667 | Fold Log Loss: 0.8015
Fold Accuracy: 0.7778 | Fold Log Loss: 0.6278
Fold Accuracy: 0.3333 | Fold Log Loss: 1.2873
Fold Accuracy: 0.7778 | Fold Log Loss: 0.7106
Fold Accuracy: 0.6667 | Fold Log Loss: 0.9280
Fold Accuracy: 0.7778 | Fold Log Loss: 0.8024
Fold Accuracy: 0.7778 | Fold Log Loss: 0.6518
Fold Accuracy: 0.4444 | Fold Log Loss: 1.1116


In [22]:

print("-" * 40)
print(f"Average Accuracy: {np.mean(scores_acc):.4f}")
print(f"Average Log Loss: {np.mean(scores_logloss):.4f}")

----------------------------------------
Average Accuracy: 0.6278
Average Log Loss: 0.9228
